# Structured RAG with SQL databse

## Download open source Chonook databse

In [6]:
import urllib.request
import os
import os

os.environ["LANGCHAIN_TRACING_V2"] = "false"

# Download Chinook database if it doesn't exist
db_path = "Chinook.db"

if not os.path.exists(db_path):
    print("Downloading Chinook database...")
    url = "https://github.com/lerocha/chinook-database/raw/master/ChinookDatabase/DataSources/Chinook_Sqlite.sqlite"
    
    try:
        urllib.request.urlretrieve(url, db_path)
        print(f"Database downloaded successfully to {db_path}")
    except Exception as e:
        print(f"Failed to download database: {e}")
        print("Please manually download the Chinook database from:")
        print("https://github.com/lerocha/chinook-database")
else:
    print(f"Database already exists at {db_path}")

# Verify the download
if os.path.exists(db_path):
    print(f"Database file size: {os.path.getsize(db_path)} bytes")
    
    # Quick verification that it has tables
    import sqlite3
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = cursor.fetchall()
    print(f"Tables in database: {[table[0] for table in tables]}")
    conn.close()

Database already exists at Chinook.db
Database file size: 1067008 bytes
Tables in database: ['Album', 'Artist', 'Customer', 'Employee', 'Genre', 'Invoice', 'InvoiceLine', 'MediaType', 'Playlist', 'PlaylistTrack', 'Track']


## Set up a LangGraph ReACT agent with SQL toolkit

In [8]:
import os
os.environ["LANGCHAIN_TRACING_V2"] = "false"

from langchain_community.agent_toolkits.sql.toolkit import SQLDatabaseToolkit
from langchain_community.utilities.sql_database import SQLDatabase
from langchain_openai import ChatOpenAI




db = SQLDatabase.from_uri("sqlite:///Chinook.db")
llm = ChatOpenAI(temperature=0)

toolkit = SQLDatabaseToolkit(db=db, llm=llm)

from langchain import hub
from langgraph.prebuilt import create_react_agent

# Pull prompt (or define your own)
prompt_template = hub.pull("langchain-ai/sql-agent-system-prompt")
system_message = prompt_template.format(dialect="SQLite", top_k=5)

# Create agent - use prompt parameter
agent_executor = create_react_agent(
    llm, toolkit.get_tools(), prompt=system_message
)


/Users/anandoka/book/bookenv/lib/python3.13/site-packages/langsmith/client.py:272: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


In [9]:
toolkit.get_tools()

[QuerySQLDatabaseTool(description="Input to this tool is a detailed and correct SQL query, output is a result from the database. If the query is not correct, an error message will be returned. If an error is returned, rewrite the query, check the query, and try again. If you encounter an issue with Unknown column 'xxxx' in 'field list', use sql_db_schema to query the correct table fields.", db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x1300b4c20>),
 InfoSQLDatabaseTool(description='Input to this tool is a comma-separated list of tables, output is the schema and sample rows for those tables. Be sure that the tables actually exist by calling sql_db_list_tables first! Example Input: table1, table2, table3', db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x1300b4c20>),
 ListSQLDatabaseTool(db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x1300b4c20>),
 QuerySQLCheckerTool(description='Use this tool to double check if your 

## Query the ReACT agent

### Simple data retrieval queries 

In [10]:
# Query agent
example_query = "Which country's customers spent the most?"

events = agent_executor.stream(
    {"messages": [("user", example_query)]},
    stream_mode="values",
)
for event in events:
    event["messages"][-1].pretty_print()

================================ Human Message =================================

Which country's customers spent the most?
================================== Ai Message ==================================
Tool Calls:
  sql_db_list_tables (call_l6mfowaNKxd7gpoYDVwA5Gnr)
 Call ID: call_l6mfowaNKxd7gpoYDVwA5Gnr
  Args:
================================= Tool Message =================================
Name: sql_db_list_tables

Album, Artist, Customer, Employee, Genre, Invoice, InvoiceLine, MediaType, Playlist, PlaylistTrack, Track
================================== Ai Message ==================================
Tool Calls:
  sql_db_schema (call_cJsrB4NzDJ9Cos0aEJ6YSSVT)
 Call ID: call_cJsrB4NzDJ9Cos0aEJ6YSSVT
  Args:
    table_names: Customer, Invoice, InvoiceLine
================================= Tool Message =================================
Name: sql_db_schema


CREATE TABLE "Customer" (
	"CustomerId" INTEGER NOT NULL, 
	"FirstName" NVARCHAR(40) NOT NULL, 
	"LastName" NVARCHAR(20) NOT NULL

In [11]:
# Query agent
example_query = "Find all customers from Brazil"

events = agent_executor.stream(
    {"messages": [("user", example_query)]},
    stream_mode="values",
)
for event in events:
    event["messages"][-1].pretty_print()

================================ Human Message =================================

Find all customers from Brazil
================================== Ai Message ==================================
Tool Calls:
  sql_db_query (call_GCbTctbgzH9DRea6eoTqRjvx)
 Call ID: call_GCbTctbgzH9DRea6eoTqRjvx
  Args:
    query: SELECT * FROM customers WHERE Country = 'Brazil'
================================= Tool Message =================================
Name: sql_db_query

Error: (sqlite3.OperationalError) no such table: customers
[SQL: SELECT * FROM customers WHERE Country = 'Brazil']
(Background on this error at: https://sqlalche.me/e/20/e3q8)
================================== Ai Message ==================================
Tool Calls:
  sql_db_list_tables (call_azzDnLoex1DcohJYpDMLRcWD)
 Call ID: call_azzDnLoex1DcohJYpDMLRcWD
  Args:
================================= Tool Message =================================
Name: sql_db_list_tables

Album, Artist, Customer, Employee, Genre, Invoice, InvoiceLin

In [12]:
# Query agent
example_query = "List the total number of tracks in each playlist."

events = agent_executor.stream(
    {"messages": [("user", example_query)]},
    stream_mode="values",
)
for event in events:
    event["messages"][-1].pretty_print()

================================ Human Message =================================

List the total number of tracks in each playlist.
================================== Ai Message ==================================
Tool Calls:
  sql_db_list_tables (call_srKWuE6KImjhjcE4CABLfxwh)
 Call ID: call_srKWuE6KImjhjcE4CABLfxwh
  Args:
================================= Tool Message =================================
Name: sql_db_list_tables

Album, Artist, Customer, Employee, Genre, Invoice, InvoiceLine, MediaType, Playlist, PlaylistTrack, Track
================================== Ai Message ==================================
Tool Calls:
  sql_db_schema (call_SID5YhxoVBlRC0SnmuV6FIf7)
 Call ID: call_SID5YhxoVBlRC0SnmuV6FIf7
  Args:
    table_names: Playlist, PlaylistTrack
================================= Tool Message =================================
Name: sql_db_schema


CREATE TABLE "Playlist" (
	"PlaylistId" INTEGER NOT NULL, 
	"Name" NVARCHAR(120), 
	PRIMARY KEY ("PlaylistId")
)

/*
3 rows from

### A complex analysis query

In [14]:
# Query agent
example_query = "Analyze sales data by employee to identify top performers and trends of their sales."

events = agent_executor.stream(
    {"messages": [("user", example_query)]},
    stream_mode="values",
)
for event in events:
    event["messages"][-1].pretty_print()

================================ Human Message =================================

Analyze sales data by employee to identify top performers and trends of their sales.
================================== Ai Message ==================================
Tool Calls:
  sql_db_list_tables (call_YT8tkHatKrmGbmE3GgfoGdQA)
 Call ID: call_YT8tkHatKrmGbmE3GgfoGdQA
  Args:
================================= Tool Message =================================
Name: sql_db_list_tables

Album, Artist, Customer, Employee, Genre, Invoice, InvoiceLine, MediaType, Playlist, PlaylistTrack, Track
================================== Ai Message ==================================
Tool Calls:
  sql_db_schema (call_a9n2GBUDOUMI7lAqCKGgg2oj)
 Call ID: call_a9n2GBUDOUMI7lAqCKGgg2oj
  Args:
    table_names: Employee, Invoice, InvoiceLine
================================= Tool Message =================================
Name: sql_db_schema


CREATE TABLE "Employee" (
	"EmployeeId" INTEGER NOT NULL, 
	"LastName" NVARCHAR(20) NO

# Unstrcutured RAG

In [16]:
from langchain import hub
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.graph import START, StateGraph
from typing_extensions import List, TypedDict

from langchain_openai import ChatOpenAI
llm = ChatOpenAI(
    model="gpt-4", # Specify the model you want to use
    temperature=0.0, # 0 for deterministic answers, higher for more creative outputs
    max_tokens=None, # More tokens for longer creative content
    timeout=None,
    max_retries=2,
    # other params...
)

# Choose an embedding model
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

# Choose a vector store - in-memory for demo purposes, but you can use others like Pinecone, Weaviate, etc.
from langchain_core.vectorstores import InMemoryVectorStore
vector_store = InMemoryVectorStore(embeddings)

# Load and chunk contents of the blog
loader = WebBaseLoader(
    web_paths=("https://www.darioamodei.com/essay/machines-of-loving-grace/",),
)
docs = loader.load()

#Split and chunk contents of the blog
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
all_splits = text_splitter.split_documents(docs)
# Index chunks
_ = vector_store.add_documents(documents=all_splits)

# Define prompt for question-answering
# N.B. for non-US LangSmith endpoints, you may need to specify
# api_url="https://api.smith.langchain.com" in hub.pull.
prompt = hub.pull("rlm/rag-prompt")


# Define state for application
class State(TypedDict):
    question: str
    context: List[Document]
    answer: str

# Define application steps
def retrieve(state: State):
    retrieved_docs = vector_store.similarity_search(state["question"])
    return {"context": retrieved_docs}


def generate(state: State):
    docs_content = "\n\n".join(doc.page_content for doc in state["context"])
    messages = prompt.invoke({"question": state["question"], "context": docs_content})
    response = llm.invoke(messages)
    return {"answer": response.content}


# Compile application and test
graph_builder = StateGraph(State).add_sequence([retrieve, generate])
graph_builder.add_edge(START, "retrieve")
graph = graph_builder.compile()

/Users/anandoka/book/bookenv/lib/python3.13/site-packages/langsmith/client.py:272: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


In [3]:
# response = graph.invoke({"question": "What is Task Decomposition?"})
import textwrap
response = graph.invoke({"question": "what is the name of the author?"})
print( textwrap.fill(response["answer"], width=80))


The name of the author is Dario Amodei.


In [ ]:
response = graph.invoke({"question": "what are the categories of AI applications that most excite Dario?"})
print(textwrap.fill(response["answer"], width=80))

The five categories of AI applications that most excite Dario are: Biology and
physical health, Neuroscience and mental health, Economic development and
poverty, Peace and governance, and Work and meaning.


In [17]:
response = graph.invoke({"question": "what is the cause of rise in MEasles in Texas in 2025?"})
print(textwrap.fill(response["answer"], width=80))

The context does not provide specific information on the cause of the rise in
Measles in Texas in 2025.
